In [5]:
import torch
import torchvision
import torchvision.transforms as transforms

# Define transformations (e.g., convert images to PyTorch tensors)
transform = transforms.ToTensor()

# Load the training dataset
train_dataset = torchvision.datasets.FashionMNIST(
    root='./data',      # Directory to store the data
    train=True,         # Load training data
    download=True,      # Download if not already available
    transform=transform
)

# Load the test dataset
test_dataset = torchvision.datasets.FashionMNIST(
    root='./data',
    train=False,        # Load test data
    download=True,
    transform=transform
)

# Use DataLoader to batch and shuffle the data
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)


100%|██████████| 26.4M/26.4M [00:03<00:00, 6.82MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 115kB/s]
100%|██████████| 4.42M/4.42M [00:02<00:00, 2.16MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 16.5MB/s]


In [7]:
class LogisticRegression:
    def __init__(self, input_dim, num_classes=2):
        self.num_classes = num_classes
        self.weights = torch.zeros(input_dim+1, num_classes, dtype=torch.float32) # each col is a class weight vector (theta_j)


    def softmax(self, X, dim=1):
        if X.dim() == 1:
            X = X.unsqueeze(0)
        # numerical stability: subtract max per row (per dim)
        exps = torch.exp(X)
        sums = exps.sum(dim=dim, keepdim=True)
        return exps / sums

    def forward(self, X):
        # X is a matrix where each row is a data point and each calumn is a feature
        #X: (batchsize ,D) tensor, weights: (D,num_classes),  logits: (batch_size, num_classes)
        if X.dim() == 1:
            X = X.unsqueeze(0)
        logits = X.matmul(self.weights) # (batvh_size, num_classes)
        return logits

    def predict_probs(self, X):
        logits = self.forward(X)
        return self.softmax(logits, dim=1)

    def predict(self, X):
        probs = self.predict_probs(X)
        return torch.argmax(probs, dim=1)

    def cross_entropy(self, logits, y):
        # logits: (N, num_classes), y: (N,) with class indices
        # each row of logits corresponds to a data point, each column corresponds to a class

        probs = self.softmax(logits, dim=1)
        print("probs", probs.shape)
        logp = torch.log(probs)
        N = logp.shape[0]
        neg_logp = -logp[torch.arange(N), y]
        print("neg_logp", neg_logp.shape)
        return torch.sum(neg_logp.unsqueeze(1))
    
    def find_gradients(self, X, y):
        # X: (N,D) tensor, y: (N,) long tensor with class indices
        logits = self.forward(X)
        probs = self.softmax(logits, dim=1)
        N = X.shape[0]
        grad_logits = probs.clone()
        grad_logits[torch.arange(N), y] -= 1  # dL/dz_j = p_j - 1 for true class, p_j for others
        grad_weights = X.t().matmul(grad_logits) / N  # (D,N) @ (N,num_classes) -> (D,num_classes)
        return grad_weights

    def fit(self, train_loader, epochs=100, lr=0.1, batch_size=None, verbose=False):

        # X: (N,D) tensor, y: (N,) long tensor with class indices
        
        losses = []
        for epoch in range(epochs):
            for batch in train_loader:
                x_batch = batch[0].view(batch[0].size(0), -1)  # Flatten the images to (batch_size, 784)
                # prepend column of ones for intercept
                ones = torch.ones(x_batch.size(0), 1, dtype=x_batch.dtype, device=x_batch.device)
                x_batch = torch.cat([ones, x_batch], dim=1)
                y_batch = batch[1].long()  
            N = x_batch.shape[0]
            epoch_loss = 0.0
            print(x_batch.shape, y_batch.shape)
            logits = self.forward(x_batch)
            print("logits", logits.shape)
            loss = self.cross_entropy(logits, y_batch)
            print("loss", loss)
            wgrad = self.find_gradients(x_batch, y_batch)
            self.weights -= lr * wgrad

            epoch_loss += loss.item() * x_batch.shape[0]

            epoch_loss /= N
            losses.append(epoch_loss)
            if verbose:
                print(f"Epoch {epoch+1}/{epochs} loss: {epoch_loss:.4f}")
        return losses
    
model = LogisticRegression(input_dim=784, num_classes=10)
model.fit(train_loader, epochs=10, lr=0.1, verbose=True)

torch.Size([32, 785]) torch.Size([32])
logits torch.Size([32, 10])
probs torch.Size([32, 10])
neg_logp torch.Size([32])
loss tensor(73.6827)
Epoch 1/10 loss: 73.6827
torch.Size([32, 785]) torch.Size([32])
logits torch.Size([32, 10])
probs torch.Size([32, 10])
neg_logp torch.Size([32])
loss tensor(69.1208)
Epoch 2/10 loss: 69.1208
torch.Size([32, 785]) torch.Size([32])
logits torch.Size([32, 10])
probs torch.Size([32, 10])
neg_logp torch.Size([32])
loss tensor(70.2461)
Epoch 3/10 loss: 70.2461
torch.Size([32, 785]) torch.Size([32])
logits torch.Size([32, 10])
probs torch.Size([32, 10])
neg_logp torch.Size([32])
loss tensor(59.2941)
Epoch 4/10 loss: 59.2941
torch.Size([32, 785]) torch.Size([32])
logits torch.Size([32, 10])
probs torch.Size([32, 10])
neg_logp torch.Size([32])
loss tensor(50.7119)
Epoch 5/10 loss: 50.7119
torch.Size([32, 785]) torch.Size([32])
logits torch.Size([32, 10])
probs torch.Size([32, 10])
neg_logp torch.Size([32])
loss tensor(55.9656)
Epoch 6/10 loss: 55.9656


KeyboardInterrupt: 